# 지도학습 vs 비지도(STFPM) 비교 — 통합 노트북

팀원 STFPM(비지도) 결과(정확도 84%)와 지도학습 결과(96~98%)의 차이가 어디서 오는지 가르기 위해, **한 번에 한 가지씩만 바꾼** 실험들을 모았다. 모두 예비 실험이다.

| 셀 | 실험 | 방식 | 입력 | 분할 |
| --- | --- | --- | --- | --- |
| 5 | ① | STFPM(팀원 코드 그대로 새로 학습) | 원본 → 고정 위치 크롭 1000 → 512 | 팀원 방식(정상 5장마다 1장 테스트, 결함 2,000장 전부 테스트) |
| 6 | ①-b | **팀원이 학습한 모델 그대로** | 같음 | ①과 같은 테스트 |
| 7 | ②' | STFPM | 같음 | 시간 블록 5-fold(전체 4,000장) |
| 8 | ③ | 지도학습 ResNet18 | 같음 | 시간 블록 5-fold(전체 4,000장) |
| 9 | ④ | STFPM | 이동·마스킹 크롭 500(자동 선별 3,425장) | 시간 블록 5-fold(3,425장) |
| 10 | ⑤ | STFPM + GPU 제한 해제(resnet34, 배치 32, epoch 50) | ②'와 같음 | ②'와 같음 |

**Drive `산공종설/`에 필요한 것** (폴더 이름이 다르면 2번 셀의 `DRIVE_DIR`을 바꾼다)
- `underfill_originals_4000.zip`: 원본 4,000장
- `keep3425_masked_500.zip`: 마스킹 크롭 3,425장 + 분할표
- `stfpm_compare_bundle_v4.zip`: 팀원 코드(`project_STFPM`, 수정 없음), 분할표, 실행 스크립트
- (6번 셀용, 선택) `wonwook_ckpt/` 폴더 안에 팀원 모델 `stfpm_model.ckpt`와 `model_metadata.json`

**실행 요령**
- 런타임은 L4를 쓴다. 노트북은 **파일 > 노트북 업로드**로 연다.
- 결과는 Drive `산공종설/results/`에 실험별 폴더로 저장된다. 이미 끝난 fold는 건너뛰므로, 끊기면 같은 셀을 다시 실행하면 이어서 돈다.
- 자리를 비울 땐 5~13번을 실행 대기시켜 둔다. 13번이 끝나면 런타임을 반납한다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > L4"

## 2. Drive 연결 + 파일 확인
`force_remount=True`는 웹에서 방금 올린 파일도 보이도록 Drive를 다시 연결한다. 이 셀이 `DRIVE_DIR`, `RESULTS_DIR` 변수를 정한다. "세션 다시 시작"을 했다면 이 셀부터 다시 실행한다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS_DIR = f"{DRIVE_DIR}/results"
for f in ("underfill_originals_4000.zip", "keep3425_masked_500.zip", "stfpm_compare_bundle_v4.zip",
          "wonwook_ckpt/stfpm_model.ckpt", "wonwook_ckpt/model_metadata.json"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 복사 → 압축 해제
Drive에서 파일을 하나씩 읽으면 느리니까 zip을 Colab 디스크로 복사해서 푼다.
- `/content/orig`: 원본 4,000장(①·②'·③·⑤)
- `/content/data`: 마스킹 크롭 3,425장과 그 분할표 `folds.csv`(④)
- `/content/bundle`: 팀원 코드와 실행 스크립트

마지막 두 줄에 4000과 3425가 나와야 한다.

In [ ]:
!rm -rf /content/bundle
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/keep3425_masked_500.zip" "{DRIVE_DIR}/stfpm_compare_bundle_v4.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/keep3425_masked_500.zip -d /content/data
!unzip -q -o /content/stfpm_compare_bundle_v4.zip -d /content/bundle
!ls /content/bundle
!find /content/orig -name "*.jpg" | wc -l
!find /content/data/size_500/masked_crops -name "*.jpg" | wc -l

## 4. anomalib 설치
팀원 코드는 이상탐지 라이브러리 `anomalib`을 쓴다. 팀원 환경과 같은 2.6.2 버전을 깐다. "세션 다시 시작" 안내가 뜨면 다시 시작한 뒤 **2~3번 셀을 다시 실행**하고 이 셀은 건너뛴다.

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md, torch
print({p: md.version(p) for p in ("anomalib", "torch", "lightning")}, "| cuda", torch.cuda.is_available())

## 5. ① STFPM, 팀원 방식 분할 (84% 재현 시도)
정상 2,000장을 프레임 순서로 세워 5장마다 1장(400장)을 테스트로 뺀다. 나머지 1,600장으로 학습하고, 언더필 2,000장은 전부 테스트로 쓴다.

`stfpm_timeblock.py`가 하는 일:
1. 분할표대로 팀원 코드가 기대하는 `train/good`, `test/good`, `test/defect` 폴더를 바로가기로 만든다.
2. 팀원 `src/train.py`를 **수정 없이** 실행한다. 학습 로그는 `foldK/train_log.txt`에 저장된다.
3. 팀원 판정기로 테스트 사진마다 점수를 매겨 `scores.csv`에 저장하고 지표를 계산한다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/stride5_split.csv --out "{RESULTS_DIR}/stfpm_resnet18_stride5"

## 6. ①-b 팀원이 학습한 모델 그대로 평가
①과 같은 테스트 사진 2,400장에 **팀원 모델 파일**을 그대로 넣는다. 팀원 판정기와 팀원 기준값(`model_metadata.json`)을 그대로 쓴다.
- 오탐(정상을 결함으로 판정)이 팀원 보고와 같으면 평가 조건이 같다는 뜻이다.
- 1차 결과: 오탐 5/400 = 1.25%로 팀원 보고와 일치했고, 결함 탐지는 26/2000이었다.

`wonwook_ckpt/`가 없으면 건너뛴다.

In [ ]:
CK = f"{DRIVE_DIR}/wonwook_ckpt"
if os.path.exists(f"{CK}/stfpm_model.ckpt"):
    !cd /content/bundle && python eval_checkpoint.py --project /content/bundle/project_STFPM --ckpt-dir "{CK}" --split-csv /content/bundle/stride5_split.csv --orig-root /content/orig --out "{RESULTS_DIR}/stfpm_wonwook_ckpt_stride5_scores.csv"
else:
    print("wonwook_ckpt 없음: 건너뜀")

## 7. ②' STFPM, 시간 블록 5-fold
①과 모델·전처리는 같고 **분할만** 바꾼다. fold마다 학습 구간의 정상 사진으로만 학습하고, 검증 구간의 정상 400 + 언더필 400으로 평가한다. 검증 사진과 30프레임 이내인 학습 사진은 뺀다(1초 간격 이웃 사진이 학습과 평가에 나뉘어 들어가는 걸 막기 위해서다).

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/folds_all4000.csv --out "{RESULTS_DIR}/stfpm_resnet18_timeblock_all4000_v2"

## 8. ③ 지도학습 ResNet18, ②'와 같은 입력·분할
**방식만** 지도학습으로 바꾼다.
- `--input-column source --fixed-crop 0 250 1000 1250`: 원본을 팀원 코드와 같은 위치(세로 0~1000, 가로 250~1250)에서 자른다. 픽셀 단위로 같은지 확인했다.
- `--img-size 512`: 팀원 설정과 같은 크기다.
- `--fast`: GPU가 가장 빠른 계산 방법을 골라 쓴다. 같은 시드로 다시 돌려도 결과가 소수점 단위로 조금 다를 수 있다.

In [ ]:
OUT3 = f"{RESULTS_DIR}/resnet18_fixedcrop512_all4000_fast_seeds0"
if os.path.exists(f"{OUT3}/summary.csv"):
    print("이미 끝남:", OUT3)
else:
    !python /content/bundle/train_pilot_classifier.py --folds-csv /content/bundle/folds_all4000.csv --pilot-root /content/orig --input-column source --fixed-crop 0 250 1000 1250 --img-size 512 --arch resnet18 --seeds 0 --fast --output "{OUT3}"

## 9. ④ STFPM + 이동·마스킹 크롭(자동 선별 3,425장)
지도학습 기존 결과(정확도 0.976, AUROC 0.992)와 **같은 사진, 같은 분할**이다.
- `--path-column masked_crop --orig-root /content/data`: 이미 잘라 둔 마스킹 크롭을 쓴다.
- `--no-roi-crop`: 이미 잘라 둔 사진이라 팀원 코드의 고정 위치 자르기를 끈다.

②'보다 좋아지면 "베드·모서리 프레임이 STFPM을 망친다"는 가설이 맞는 거다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/data --path-column masked_crop --no-roi-crop --split-csv /content/data/folds.csv --out "{RESULTS_DIR}/stfpm_resnet18_keep3425_masked"

## 10. ⑤ STFPM + GPU 제한 해제
②'와 사진·분할이 같고, 팀원 `train.py`에 들어 있던 GTX 1660용 제한만 푼다.
- `--relax`: 팀원 코드를 **복사한 사본**에서 네 줄만 바꾼다(백본 제한 1줄, 배치 16 상한 2줄, epoch 20 상한 1줄). 원본은 그대로다.
- `CUDA out of memory`가 나면 `--batch-size 16`으로 줄인다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/folds_all4000.csv --relax --backbone resnet34 --batch-size 32 --max-epochs 50 --out "{RESULTS_DIR}/stfpm_resnet34_b32_e50_timeblock_all4000"

## 11. 비교표
- `auroc`: 판정 기준과 상관없이 두 클래스를 얼마나 잘 가르는지(0.5 = 찍기, 1 = 완벽). STFPM은 fold별 AUROC의 평균이다. fold마다 모델이 달라 점수 눈금이 다르기 때문이다.
- `accuracy`, `underfill_recall`, `normal_recall`: STFPM은 팀원 방식 기준값(정상 학습 점수 상위 1%)으로, 지도학습은 확률 0.5로 판정한 결과다.
- 끝난 실험만 표에 나온다.

In [ ]:
import json, pandas as pd
from sklearn.metrics import roc_auc_score
rows, folds = [], {}
def stfpm(name, sub):
    s = json.load(open(f"{RESULTS_DIR}/{sub}/summary.json"))
    p = s["pooled"]
    rows.append({"run": name, "accuracy": p["accuracy"], "underfill_recall": p["underfill_recall"],
                 "normal_recall": p["normal_recall"], "auroc": s["fold_mean_auroc"], "n_test": p["n"]})
    folds[name] = {k: round(v["auroc"], 3) for k, v in s["per_fold"].items()}
def ckpt_scores(name, path):
    s = pd.read_csv(path, encoding="utf-8-sig")
    y, p = s["label_id"], s["pred"]
    rows.append({"run": name, "accuracy": (y == p).mean(), "underfill_recall": p[y == 1].mean(),
                 "normal_recall": 1 - p[y == 0].mean(), "auroc": roc_auc_score(y, s["score"]), "n_test": len(s)})
def supervised(name, sub):
    s = pd.read_csv(f"{RESULTS_DIR}/{sub}/summary.csv", encoding="utf-8-sig")
    for _, r in s.iterrows():
        rows.append({"run": f"{name} ({r['mode']})", "accuracy": r["pooled_mean_accuracy"],
                     "underfill_recall": r["pooled_mean_underfill_recall"], "normal_recall": r["pooled_mean_normal_recall"],
                     "auroc": r["pooled_mean_auroc"], "n_test": None})
jobs = (("① STFPM 팀원분할", "stfpm_resnet18_stride5", stfpm),
        ("①-b 팀원 모델 그대로", f"{RESULTS_DIR}/stfpm_wonwook_ckpt_stride5_scores.csv", ckpt_scores),
        ("②' STFPM 시간블록", "stfpm_resnet18_timeblock_all4000_v2", stfpm),
        ("④ STFPM 마스킹·선별", "stfpm_resnet18_keep3425_masked", stfpm),
        ("⑤ STFPM GPU제한해제", "stfpm_resnet34_b32_e50_timeblock_all4000", stfpm),
        ("③ 지도 고정크롭", "resnet18_fixedcrop512_all4000_fast_seeds0", supervised),
        ("지도 마스킹·선별", "resnet18_keep3425_masked_500_fast_seeds0", supervised))
for name, sub, fn in jobs:
    try:
        fn(name, sub)
    except FileNotFoundError:
        print("아직 없음:", name)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSTFPM fold별 AUROC (fold 0 = 출력 초반):")
for k, v in folds.items():
    print(" ", k, v)

## 12. 기준값만 잘 잡으면 STFPM은 어디까지 가나
STFPM은 AUROC가 괜찮아도 기본 기준값(정상 점수 상위 1%)이 너무 보수적이라 결함을 거의 못 잡는다. 기준값을 바꾸면 어떻게 되는지 fold마다 계산해서 평균낸다.
- `오탐X%일때_탐지율`: 정상의 X%를 결함으로 잘못 잡는 것까지 허용할 때, 언더필을 몇 % 잡는지.
- `최고_균형정확도(상한)`: 테스트 정답을 보고 가장 좋은 기준값을 고른 값이다. **실제보다 낙관적인 상한선**으로만 본다.

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve
for name, sub in (("②' 고정크롭", "stfpm_resnet18_timeblock_all4000_v2"),
                  ("④ 마스킹·선별", "stfpm_resnet18_keep3425_masked"),
                  ("⑤ GPU제한해제", "stfpm_resnet34_b32_e50_timeblock_all4000")):
    out = []
    for k in range(5):
        p = f"{RESULTS_DIR}/{sub}/fold{k}/scores.csv"
        if not os.path.exists(p):
            break
        s = pd.read_csv(p, encoding="utf-8-sig")
        fpr, tpr, _ = roc_curve(s["label_id"], s["score"])   # 기준값을 바꿔 가며 (오탐률, 탐지율) 쌍을 모두 계산
        r = {f"오탐{int(f * 100)}%일때_탐지율": np.interp(f, fpr, tpr) for f in (0.01, 0.05, 0.10)}
        r["최고_균형정확도(상한)"] = ((tpr + 1 - fpr) / 2).max()
        out.append(r)
    if len(out) == 5:
        print(name, pd.DataFrame(out).mean().round(3).to_dict())
    else:
        print(name, ": 아직 5 fold가 다 끝나지 않음")

## 13. 런타임 반납
Colab 사용량은 GPU 런타임에 연결된 시간만큼 차감된다. 자리를 비울 땐 이 셀도 실행 대기시켜 둔다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # Drive에 덜 올라간 파일을 마저 올리고 연결 해제
runtime.unassign()          # 런타임 반납